In [1]:
import pandas as pd
import random

# ============================================================
# PATCH-LEVEL TRAIN/VAL/TEST SPLIT — stratified by tier
# 15 patches total: 9 train / 3 val / 3 test (3/1/1 per tier)
# ============================================================
patch_tiers = {
    "HIGH": ['0,6', '0,0', '1,0', '3,4', '0,2'],
    "MEAN": ['5,0', '5,1', '6,0', '4,1', '4,3'],
    "LOW":  ['2,4', '1,3', '3,1', '6,1', '4,6'],
}

RANDOM_SEED = 42  # fixed for reproducibility — change if you want a different split
random.seed(RANDOM_SEED)

split_assignment = {}  # patch_id -> split

for tier, patches in patch_tiers.items():
    shuffled = patches.copy()
    random.shuffle(shuffled)
    train_patches = shuffled[:3]
    val_patches = shuffled[3:4]
    test_patches = shuffled[4:5]

    for p in train_patches:
        split_assignment[p] = "train"
    for p in val_patches:
        split_assignment[p] = "val"
    for p in test_patches:
        split_assignment[p] = "test"

    print(f"{tier}: train={train_patches} val={val_patches} test={test_patches}")

print("\nFull assignment:")
for patch_id, split in split_assignment.items():
    print(f"  {patch_id}: {split}")

split_df = pd.DataFrame([
    {"patch_id": k, "split": v} for k, v in split_assignment.items()
])
split_df.to_parquet(r"processed/patch_split_assignment.parquet", index=False)
print("\nSaved: processed/patch_split_assignment.parquet")


# ============================================================
# APPLY SPLIT TO BOTH MANIFESTS
# ============================================================
manifest = pd.read_parquet(r"processed/tile_manifest.parquet")
manifest_tf = pd.read_parquet(r"processed/tile_manifest_tf.parquet")

manifest["split"] = manifest["patch_id"].map(split_assignment)
manifest_tf["split"] = manifest_tf["patch_id"].map(split_assignment)

print("\n=== TILES split tile counts ===")
print(manifest.groupby("split").size())
print("\n=== TILES_TF split tile counts ===")
print(manifest_tf.groupby("split").size())

manifest.to_parquet(r"processed/tile_manifest.parquet", index=False)
manifest_tf.to_parquet(r"processed/tile_manifest_tf.parquet", index=False)
print("\nUpdated both manifests with 'split' column, saved in place.")

HIGH: train=['3,4', '0,0', '1,0'] val=['0,2'] test=['0,6']
MEAN: train=['4,1', '6,0', '5,0'] val=['4,3'] test=['5,1']
LOW: train=['6,1', '1,3', '3,1'] val=['2,4'] test=['4,6']

Full assignment:
  3,4: train
  0,0: train
  1,0: train
  0,2: val
  0,6: test
  4,1: train
  6,0: train
  5,0: train
  4,3: val
  5,1: test
  6,1: train
  1,3: train
  3,1: train
  2,4: val
  4,6: test

Saved: processed/patch_split_assignment.parquet

=== TILES split tile counts ===
split
test      4058
train    13246
val       4862
dtype: int64

=== TILES_TF split tile counts ===
split
test      5135
train    16875
val       6131
dtype: int64

Updated both manifests with 'split' column, saved in place.
